In [ ]:
NOTEBOOK_TITLE = '099: DueCare Orientation + Setup Conclusion'
from IPython.display import HTML, display
display(HTML(
    '<div style="background:linear-gradient(135deg,#1e3a8a 0%,#4c78a8 100%);color:white;padding:20px 24px;border-radius:8px;margin:8px 0;font-family:system-ui,-apple-system,sans-serif">'
    '<div style="font-size:10px;font-weight:600;letter-spacing:0.14em;opacity:0.8;text-transform:uppercase">DueCare - Section Conclusion</div>'
    f'<div style="font-size:22px;font-weight:700;margin:4px 0 0 0">{NOTEBOOK_TITLE}</div>'
    '<div style="font-size:13px;opacity:0.92;margin-top:4px">Recap, key findings, and handoff to the next section.</div></div>'
))


In [ ]:
# Install the pinned DueCare packages for this notebook.
import glob
import subprocess
import sys

PACKAGES = ['duecare-llm-core==0.1.0']
DUECARE_PACKAGES = [package for package in PACKAGES if package.startswith('duecare-')]
EXTRA_PACKAGES = [package for package in PACKAGES if not package.startswith('duecare-')]

def _pip_install(items):
    if items:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *items])

def _wheel_install():
    wheel_patterns = [
        '/kaggle/input/duecare-llm-wheels/*.whl',
        '/kaggle/input/datasets/taylorsamarel/duecare-llm-wheels/*.whl',
        '/kaggle/input/**/*.whl',
    ]
    wheel_files = []
    for pattern in wheel_patterns:
        wheel_files = sorted(glob.glob(pattern, recursive=True))
        if wheel_files:
            break
    if not wheel_files:
        return False
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', *wheel_files])
    if EXTRA_PACKAGES:
        _pip_install(EXTRA_PACKAGES)
    print(f'Installed {len(wheel_files)} wheel files via attached Kaggle dataset.')
    return True

def _duecare_importable():
    try:
        import importlib
        for mod in ('duecare.core',):
            importlib.invalidate_caches()
            importlib.import_module(mod)
        return True
    except Exception:
        return False

install_source = 'pypi'
try:
    _pip_install(PACKAGES)
except Exception as exc:
    print(f'Pinned PyPI install failed ({exc.__class__.__name__}). Falling back to Kaggle wheels for DueCare packages.')
    if not _wheel_install() and DUECARE_PACKAGES:
        raise RuntimeError('Could not install pinned DueCare packages from PyPI or attached wheel datasets.') from exc
    install_source = 'kaggle_wheels'
else:
    # PyPI pip install returned 0 but that does not guarantee `duecare` is
    # importable (a stub package with the same name can satisfy pip while
    # providing none of the real modules). Verify; fall back to wheels if
    # the import is still broken.
    if DUECARE_PACKAGES and not _duecare_importable():
        print('PyPI install finished but duecare.core is not importable. Falling back to Kaggle wheels.')
        if _wheel_install():
            install_source = 'kaggle_wheels'
        else:
            raise RuntimeError('duecare.core not importable after pip and wheel install. Attach taylorsamarel/duecare-llm-wheels and rerun.')

import duecare.core
print(f'DueCare version: {duecare.core.__version__} ({install_source})')
if duecare.core.__version__ != '0.1.0':
    print('Expected DueCare version: 0.1.0')


# 099: DueCare Orientation and Background and Package Setup Conclusion

This notebook closes the **Orientation and Background and Package Setup** section of the DueCare suite. Open it after you have worked through the section's notebooks in order. It recaps what the section established and hands off to the next narrative step.

<table style="width: 100%; border-collapse: collapse; margin: 4px 0 8px 0;">
  <thead>
    <tr style="background: #f6f8fa; border-bottom: 2px solid #d1d5da;">
      <th style="padding: 6px 10px; text-align: left; width: 22%;">Field</th>
      <th style="padding: 6px 10px; text-align: left; width: 78%;">Value</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px 10px;"><b>Inputs</b></td><td style="padding: 6px 10px;">Notebooks read in this section (000 Index, 005 Glossary and Reading Map, 010 Quickstart).</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Outputs</b></td><td style="padding: 6px 10px;">A plain-English recap, key points, and the next-section handoff.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Prerequisites</b></td><td style="padding: 6px 10px;">Kaggle CPU kernel with internet enabled; no GPU or API keys.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Runtime</b></td><td style="padding: 6px 10px;">Under 1 minute.</td></tr>
    <tr><td style="padding: 6px 10px;"><b>Pipeline position</b></td><td style="padding: 6px 10px;">Orientation and Background and Package Setup section close. Previous: <a href="https://www.kaggle.com/code/taylorsamarel/010-duecare-quickstart-in-5-minutes">010 Quickstart</a>. Next section: <a href="https://www.kaggle.com/code/taylorsamarel/duecare-real-gemma-4-on-50-trafficking-prompts">100 Gemma Exploration</a>.</td></tr>
  </tbody>
</table>

---

## Recap

The orientation section did three things, in this order. First, 000 Index laid out the full suite as eleven sections with an explicit reading order, so a reader lands on any later notebook knowing where it sits in the narrative rather than treating it as a standalone demo. Second, 005 Glossary and Reading Map named the vocabulary the rest of the suite uses (DueCare, Gemma 4, 6-dimension weighted rubric, V3 6-band classifier, domain packs, Anonymizer gate, cross-domain proof) and gave each term a concrete pointer to the notebook that introduces it. Third, 010 Quickstart installed the 8 duecare-llm-* PyPI packages, verified the model, task, agent, and domain registries resolved with real plugins (not placeholders), and ran a minimal safety-scoring loop end-to-end on a free Kaggle CPU kernel. Together, the section converts an abstract project description into a working install that any reader can reproduce in five minutes before any GPU-bound evaluation section spends compute.

**Section notebooks covered:** 000 Index, 005 Glossary and Reading Map, 010 Quickstart.

---

## Key points

- DueCare is named for California Civil Code section 1714(a) duty of care; the duty-of-care framing is the legal basis for why NGOs need an on-device evaluator rather than a frontier API.
- The project is an applied exploration of Gemma 4 as an on-device safety system; Gemma 4's native function calling and multimodal understanding are load-bearing infrastructure, not demo-only decoration.
- The suite ships as 8 PyPI packages under the duecare-llm-* namespace sharing the duecare Python import namespace (PEP 420), so a Kaggle notebook can pip install only the subset it needs instead of multi-GB of deps.
- The registries (8 model adapters, 9 capability tests, 12 agents, 3 domain packs) auto-register on import; the quickstart asserts non-empty registries so every later section has real plugins to call.
- The glossary and reading map are the navigation surface; a reader who lands on a single notebook via a public link can still orient themselves because every later notebook's header table names its inputs, outputs, prerequisites, runtime, and pipeline position.
- With the environment verified, the next section can spend GPU time on real Gemma 4 inference instead of re-proving the install path works.

---

## Where to go next

- **Continue to the next section:** [100 Gemma Exploration](https://www.kaggle.com/code/taylorsamarel/duecare-real-gemma-4-on-50-trafficking-prompts).
- **Back to navigation (optional):** [000 Index](https://www.kaggle.com/code/taylorsamarel/duecare-000-index).


In [ ]:
print(
    'Section close complete. Continue to 100 Gemma Exploration: '
    'https://www.kaggle.com/code/taylorsamarel/duecare-real-gemma-4-on-50-trafficking-prompts'
    '.'
)
